<a href="https://colab.research.google.com/github/satyamroy3913-maker/CurrencyConverters/blob/main/Day02_Goal01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, sys, re, time, json, subprocess, getpass
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai", "pandas"], check=True)

import pandas as pd
from collections import Counter
from google import genai
from google.genai import types

DELAY = 6.5   # free tier ke liye. Paid key ho to 0.5

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    pass
api_key = api_key or os.environ.get("GEMINI_API_KEY") or getpass.getpass("Gemini API key paste karo: ")
client = genai.Client(api_key=api_key)

CANDIDATES = ["gemini-flash-lite-latest", "gemini-2.5-flash-lite", "gemini-flash-latest", "gemini-2.5-flash"]

def pick_working_model(candidates):
    for name in candidates:
        for attempt in range(2):
            try:
                r = client.models.generate_content(model=name, contents="Reply with the word OK.")
                if r.text:
                    return name
            except Exception as e:
                print(f"  {name}: {str(e)[:70]}... (try {attempt+1}/2)")
                time.sleep(8)
    return None

MODEL = pick_working_model(CANDIDATES)
if MODEL is None:
    raise RuntimeError("Abhi koi model respond nahi kar raha. 15-20 min baad dobara run karo.")
print("Using model:", MODEL)

In [ ]:
COT_DIR = "/content/bootcamp-repo/cot"
PROMPT_DIR = os.path.join(COT_DIR, "prompts")
os.makedirs(PROMPT_DIR, exist_ok=True)
print("Folders ready:", COT_DIR)

In [ ]:
SCEN = [
 {"id": 1, "answer": "No", "scenario":
  "Customer: Anita Rao. Policy: health insurance, start date 1 Jan 2024.\n"
  "Rules: (1) Illness-related hospitalisation is covered only after a 90-day waiting period from the policy start date. (2) Accident-related hospitalisation is covered from day 1.\n"
  "Claim: Anita was hospitalised for viral fever on 15 Feb 2024."},
 {"id": 2, "answer": "Yes", "scenario":
  "Customer: Sanjay Kulkarni. Policy: health insurance, start date 1 Jan 2024.\n"
  "Rules: (1) Illness-related hospitalisation is covered only after a 90-day waiting period from the policy start date. (2) Accident-related hospitalisation is covered from day 1.\n"
  "Claim: Sanjay was hospitalised for appendicitis surgery on 15 Jun 2024."},
 {"id": 3, "answer": "No", "scenario":
  "Customer: Mohan Das. Policy: motor insurance, annual premium due 1 Mar 2025.\n"
  "Rules: (1) A premium unpaid for more than 15 days after the due date causes the policy to lapse. (2) After a lapse, coverage restarts only from the date the premium is actually paid.\n"
  "Claim: Mohan paid the premium on 25 Mar 2025. His car was damaged in an accident on 20 Mar 2025."},
 {"id": 4, "answer": "Yes", "scenario":
  "Customer: Geeta Pillai. Policy: motor insurance, annual premium due 1 Mar 2025.\n"
  "Rules: (1) A premium unpaid for more than 15 days after the due date causes the policy to lapse. (2) After a lapse, coverage restarts only from the date the premium is actually paid. (3) If the premium is paid within the 15-day grace period, coverage is treated as continuous.\n"
  "Claim: Geeta paid the premium on 10 Mar 2025. Her car was damaged in an accident on 5 Mar 2025."},
 {"id": 5, "answer": "No", "scenario":
  "Customer: Ravi Menon. Policy: car insurance.\n"
  "Rules: (1) A deductible of Rs 10,000 is subtracted from every repair claim. (2) If the payout after the deductible is below Rs 5,000, the claim is rejected.\n"
  "Claim: Repair bill of Rs 14,000 for collision damage. All the damage is covered."},
 {"id": 6, "answer": "Yes", "scenario":
  "Customer: Sheela Iyer. Policy: car insurance.\n"
  "Rules: (1) A deductible of Rs 10,000 is subtracted from every repair claim. (2) If the payout after the deductible is below Rs 5,000, the claim is rejected.\n"
  "Claim: Repair bill of Rs 16,500 for collision damage. All the damage is covered."},
 {"id": 7, "answer": "No", "scenario":
  "Customer: Kavita Joshi. Policy: home insurance, sum insured Rs 5,00,000.\n"
  "Rules: (1) The total claim amount for one incident must not exceed the sum insured; claims above it are rejected. (2) Fire damage repair and temporary accommodation costs both count towards the total.\n"
  "Claim: Fire damage repair Rs 4,20,000 and temporary accommodation Rs 1,10,000 for the same incident."},
 {"id": 8, "answer": "No", "scenario":
  "Customer: Arjun Bhatt. Policy: travel insurance.\n"
  "Rules: (1) Injuries from adventure sports (paragliding, scuba diving, bungee jumping) are excluded unless the Adventure Add-on was purchased. (2) All other accidental injuries abroad are covered.\n"
  "Claim: Arjun broke his leg while paragliding in Bali. He did not purchase the Adventure Add-on."},
 {"id": 9, "answer": "Yes", "scenario":
  "Customer: Neha Kulkarni. Policy: travel insurance.\n"
  "Rules: (1) Injuries from adventure sports (paragliding, scuba diving, bungee jumping) are excluded unless the Adventure Add-on was purchased. (2) All other accidental injuries abroad are covered. (3) The Adventure Add-on covers treatment costs up to Rs 1,00,000.\n"
  "Claim: Neha injured her ear while scuba diving in the Maldives. She purchased the Adventure Add-on. Treatment cost Rs 80,000."},
 {"id": 10, "answer": "No", "scenario":
  "Customer: Imran Qureshi. Policy: health insurance.\n"
  "Rules: Claims must be filed within 30 days after the incident date. Claims filed later are rejected.\n"
  "Claim: The incident happened on 2 Apr 2025 and Imran filed the claim on 5 May 2025."},
 {"id": 11, "answer": "Yes", "scenario":
  "Customer: Pooja Verma. Policy: health insurance.\n"
  "Rules: Claims must be filed within 30 days after the incident date. Claims filed later are rejected.\n"
  "Claim: The incident happened on 20 Jan 2025 and Pooja filed the claim on 18 Feb 2025."},
 {"id": 12, "answer": "No", "scenario":
  "Customer: Sunita Rao. Policy: health insurance.\n"
  "Rules: Hospitalisation is covered only if the patient was admitted for at least 24 continuous hours. Shorter stays (day-care treatments) are not covered.\n"
  "Claim: Sunita was admitted on 10 Oct 2025 at 9:00 PM and discharged on 11 Oct 2025 at 6:00 PM."},
 {"id": 13, "answer": "Yes", "scenario":
  "Customer: Harish Gupta. Policy: health insurance.\n"
  "Rules: Hospitalisation is covered only if the patient was admitted for at least 24 continuous hours. Shorter stays (day-care treatments) are not covered.\n"
  "Claim: Harish was admitted on 3 Nov 2025 at 8:00 AM and discharged on 4 Nov 2025 at 10:00 AM."},
 {"id": 14, "answer": "No", "scenario":
  "Customer: Deepak Nair. Policy: motor insurance, active through 31 Dec 2025.\n"
  "Rules: A claim is approved only if both conditions hold on the accident date: (a) the policy is active, and (b) the driver has a valid driving licence.\n"
  "Claim: Accident on 15 Aug 2025. The driver's licence expired on 1 Aug 2025 and was not renewed."},
 {"id": 15, "answer": "Yes", "scenario":
  "Customer: Farah Khan. Policy: health insurance, start date 1 Mar 2024.\n"
  "Rules: Conditions diagnosed before the policy start date are excluded for the first 2 years of the policy. After 2 years they are covered.\n"
  "Claim: Hospitalisation for diabetes complications on 10 Mar 2026. Diabetes was diagnosed in 2022."},
]
assert len(SCEN) == 15
print("Yes:", sum(s["answer"] == "Yes" for s in SCEN), "| No:", sum(s["answer"] == "No" for s in SCEN))

with open(os.path.join(COT_DIR, "scenarios.jsonl"), "w", encoding="utf-8") as f:
    for s in SCEN:
        f.write(json.dumps({"scenario_id": s["id"], "scenario": s["scenario"],
                            "question": "Should the claim be approved? Yes/No",
                            "ground_truth": s["answer"]}, ensure_ascii=False) + "\n")
print("scenarios.jsonl saved")

In [ ]:
DIRECT_P = """You are an insurance claims reviewer. Read the scenario and decide whether the claim should be approved.

Scenario:
{scenario}

Answer Yes or No only."""

COT_P = """You are an insurance claims reviewer. Read the scenario and decide whether the claim should be approved.

Scenario:
{scenario}

Think step by step: check each rule against the facts and do any date, hour or amount calculations explicitly. Then decide.
End your reply with a final line in exactly this format:
Final answer: Yes
or
Final answer: No"""

SC_P = COT_P   # same CoT prompt, run 5 times at temperature 0.7, then majority vote

for name, text in {"direct": DIRECT_P, "cot": COT_P, "sc": SC_P}.items():
    with open(os.path.join(PROMPT_DIR, f"{name}.txt"), "w") as f:
        f.write(text)
print("Prompts saved:", os.listdir(PROMPT_DIR))

Step 5

In [ ]:
PRICE_IN = 0.10    # USD per 1M input tokens (assumed). Current price check karke badal lo.
PRICE_OUT = 0.40   # USD per 1M output tokens (assumed)
NO_THINK = True

def ask(prompt, temperature):
    """Returns (text, cost_usd, tokens_in, tokens_out). 503 par retry, 400 par setting hata ke dobara try."""
    global NO_THINK
    for attempt in range(8):
        try:
            cfg = {"temperature": temperature}
            if NO_THINK:
                cfg["thinking_config"] = types.ThinkingConfig(thinking_budget=0)
            r = client.models.generate_content(
                model=MODEL, contents=prompt, config=types.GenerateContentConfig(**cfg))
            u = r.usage_metadata
            t_in = (u.prompt_token_count or 0) if u else 0
            t_out = ((u.candidates_token_count or 0) + (getattr(u, "thoughts_token_count", 0) or 0)) if u else 0
            cost = (t_in * PRICE_IN + t_out * PRICE_OUT) / 1_000_000
            return (r.text or ""), cost, t_in, t_out
        except Exception as e:
            msg = str(e)
            if "400" in msg or "INVALID_ARGUMENT" in msg:
                if NO_THINK:
                    NO_THINK = False
                    print("  (is model pe thinking_budget=0 accept nahi hua, bina uske chala raha hu)")
                    continue
                raise RuntimeError(f"400 error bina thinking setting ke bhi aaya, poora error dekho:\n{msg}")
            wait = min(15 * (attempt + 1), 90)
            print(f"  API error ({msg[:70]}...) - {wait}s baad retry")
            time.sleep(wait)
    raise RuntimeError("Model abhi busy hai. 10-15 min baad dobara run karo.")

def parse_direct(text):
    m = re.search(r"\b(yes|no)\b", text, re.I)
    return m.group(1).capitalize() if m else "Invalid"

def parse_cot(text):
    hits = re.findall(r"final answer:\s*\**\s*(yes|no)", text, re.I)
    return hits[-1].capitalize() if hits else "Invalid"

print("Helpers ready | thinking off try:", NO_THINK)

Step 6:Step 3a, Direct prompt

In [ ]:
DIRECT = []
for s in SCEN:
    text, cost, ti, to = ask(DIRECT_P.replace("{scenario}", s["scenario"]), temperature=0)
    ans = parse_direct(text)
    DIRECT.append({"id": s["id"], "answer": ans, "raw": text, "cost": cost, "tokens_out": to})
    print(f"[{s['id']:02d}/15] direct -> {ans} (truth: {s['answer']})")
    time.sleep(DELAY)

Cell 7: Step 3b, CoT prompt

In [ ]:
COT = []
for s in SCEN:
    text, cost, ti, to = ask(COT_P.replace("{scenario}", s["scenario"]), temperature=0)
    ans = parse_cot(text)
    COT.append({"id": s["id"], "answer": ans, "raw": text, "cost": cost, "tokens_out": to})
    print(f"[{s['id']:02d}/15] cot -> {ans} (truth: {s['answer']}) | output tokens: {to}")
    time.sleep(DELAY)

Cell 8: Step 3c, Self-consistency

In [ ]:
SC = []
for s in SCEN:
    votes, raws, cost_sum = [], [], 0.0
    for k in range(5):
        text, cost, ti, to = ask(SC_P.replace("{scenario}", s["scenario"]), temperature=0.7)
        votes.append(parse_cot(text))
        raws.append(text)
        cost_sum += cost
        time.sleep(DELAY)
    valid = [v for v in votes if v != "Invalid"]
    majority = Counter(valid).most_common(1)[0][0] if valid else "Invalid"
    SC.append({"id": s["id"], "answer": majority, "votes": votes, "raws": raws, "cost": cost_sum})
    print(f"[{s['id']:02d}/15] sc votes {votes} -> {majority} (truth: {s['answer']})")

In [ ]:
rows = []
for s, d, c, v in zip(SCEN, DIRECT, COT, SC):
    rows.append({
        "scenario_id": s["id"],
        "direct_answer": d["answer"], "direct_correct": d["answer"] == s["answer"],
        "cot_answer": c["answer"],    "cot_correct": c["answer"] == s["answer"],
        "sc_answer": v["answer"],     "sc_correct": v["answer"] == s["answer"],
        "direct_cost_usd": round(d["cost"], 6),
        "cot_cost_usd": round(c["cost"], 6),
        "sc_cost_usd": round(v["cost"], 6),
    })
df = pd.DataFrame(rows)
df.to_csv(os.path.join(COT_DIR, "results.csv"), index=False)

with open(os.path.join(COT_DIR, "results_details.json"), "w", encoding="utf-8") as f:
    json.dump({"direct": DIRECT, "cot": COT, "sc": SC}, f, indent=2, ensure_ascii=False)

summary = {}
for key, label in [("direct", "Direct"), ("cot", "CoT"), ("sc", "Self-consistency")]:
    correct = int(df[f"{key}_correct"].sum())
    total_cost = df[f"{key}_cost_usd"].sum()
    summary[label] = {"correct": correct, "acc": correct / len(df) * 100, "cost": total_cost,
                      "cpc": (total_cost / correct) if correct else None}
for label, m in summary.items():
    cpc = f"${m['cpc']:.6f}" if m["cpc"] is not None else "n/a"
    print(f"{label:17s}: {m['correct']}/15 = {m['acc']:.0f}% | total ${m['cost']:.6f} | per correct {cpc}")
df

In [ ]:
d, c, s = summary["Direct"], summary["CoT"], summary["Self-consistency"]
def cpc(m): return f"${m['cpc']:.6f}" if m["cpc"] is not None else "n/a"

wrong_direct_right_cot = df[(~df["direct_correct"]) & (df["cot_correct"])]["scenario_id"].tolist()
wrong_cot_right_sc = df[(~df["cot_correct"]) & (df["sc_correct"])]["scenario_id"].tolist()
still_wrong_all = df[(~df["direct_correct"]) & (~df["cot_correct"]) & (~df["sc_correct"])]["scenario_id"].tolist()

md = ["# CoT Decision Rule: Direct vs Chain-of-Thought vs Self-Consistency\n",
      f"Model: `{MODEL}` | Scenarios: 15 (insurance claim approval, Yes/No) | Direct & CoT: temperature 0 | Self-consistency: CoT x5 at temperature 0.7, majority vote\n",
      "## Accuracy and cost\n",
      "| Method | Correct | Accuracy | Total cost (USD, est.) | Cost per correct answer |",
      "|---|---|---|---|---|",
      f"| Direct | {d['correct']}/15 | {d['acc']:.0f}% | ${d['cost']:.6f} | {cpc(d)} |",
      f"| CoT | {c['correct']}/15 | {c['acc']:.0f}% | ${c['cost']:.6f} | {cpc(c)} |",
      f"| Self-consistency | {s['correct']}/15 | {s['acc']:.0f}% | ${s['cost']:.6f} | {cpc(s)} |",
      "",
      f"- Direct wrong but CoT right: {wrong_direct_right_cot or 'none'}",
      f"- CoT wrong but self-consistency right: {wrong_cot_right_sc or 'none'}",
      f"- Wrong in all three methods: {still_wrong_all or 'none'}",
      "",
      "_Cost is an estimate from token counts using assumed per-token prices. 15 scenarios is a small set, so differences of 1-2 answers are indicative, not proof._\n",
      "## What the numbers say\n"]

if c["acc"] > d["acc"]:
    md.append(f"- CoT raised accuracy from {d['acc']:.0f}% to {c['acc']:.0f}% at about {c['cost']/d['cost']:.1f}x the cost of direct." if d["cost"] else "- CoT raised accuracy over direct.")
else:
    md.append(f"- CoT did not beat direct here ({c['acc']:.0f}% vs {d['acc']:.0f}%), so its extra cost (about {c['cost']/d['cost']:.1f}x) bought nothing on this set." if d["cost"] else "- CoT did not beat direct here.")
if s["acc"] > c["acc"]:
    md.append(f"- Self-consistency raised accuracy from {c['acc']:.0f}% to {s['acc']:.0f}% at about {s['cost']/c['cost']:.1f}x the cost of CoT." if c["cost"] else "- Self-consistency raised accuracy over CoT.")
else:
    md.append(f"- Self-consistency did not beat CoT here ({s['acc']:.0f}% vs {c['acc']:.0f}%) while costing about {s['cost']/c['cost']:.1f}x more." if c["cost"] else "- Self-consistency did not beat CoT here.")

md += ["\n## Decision rule\n",
       "- **Use direct** when the decision is a single lookup or one simple rule, speed and cost matter, and the model already gets it right.",
       "- **Use CoT** when the answer needs several steps: date or amount arithmetic, more than one rule, exceptions. The reasoning also gives an audit trail you can show a reviewer.",
       "- **Use self-consistency** only when a wrong answer is expensive (large claim, legal or financial impact) and CoT answers change between runs. Treat it as a 5x cost for a safety margin, not a default.",
       "- Start from direct, test on real cases, and move up one level only if the measured accuracy gain justifies the extra cost per correct answer.",
       "\n## My observations (apne words me likho)\n",
       "- Which scenarios did direct get wrong, and what kind of reasoning did they need (dates, amounts, hours)?",
       "- Were the CoT votes in self-consistency usually unanimous? What does that say about its value here?",
       "- For a real insurance company, which method would you pick and why?"]

with open(os.path.join(COT_DIR, "cot_decision_rule.md"), "w") as f:
    f.write("\n".join(md))
print("cot_decision_rule.md saved")
print("\n".join(md[:14]))

Cell 11: Files check + download

In [ ]:
for root, dirs, files_ in os.walk(COT_DIR):
    level = root.replace(COT_DIR, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for fn in sorted(files_):
        print("  " * (level + 1) + fn)

from google.colab import files
!cd /content/bootcamp-repo && zip -r cot.zip cot
files.download("/content/bootcamp-repo/cot.zip")